In [1]:
import numpy as np

def get_reward(action, true_goal):
    g1_effort, g2_effort = action
    favor_g1 = g1_effort - g2_effort

    if true_goal == 'g1':
        return 1 if favor_g1 > 0 else 0 if favor_g1 == 0 else -1
    else:
        return 1 if favor_g1 < 0 else 0 if favor_g1 == 0 else -1




In [9]:
movements = {
    'up': (-1, 0),
    'down': (1, 0),
    'left': (0, -1),
    'right': (0, 1),
    'none': (0, 0)  # This represents no movement
}


In [10]:
def update_position(position, mover_action):
    N = 7  # Grid size for a 7x7 grid
    # Translate mover action to grid movement
    movement = movements.get(mover_action, (0, 0))  # Defaults to no movement if action not recognized
    new_position = (
        max(0, min(N - 1, position[0] + movement[0])),  # Ensure the new position does not go out of grid bounds
        max(0, min(N - 1, position[1] + movement[1]))
    )
    return new_position


In [23]:
def env(state, action, mover_action):
    N = 7  # Grid size
    position, bananas_consumed_at_g1, bananas_consumed_at_g2 = state

    # Updated goal positions could be static or dynamic
    g1_position = (2, 5)  # g1's position could change if the game rules say so
    g2_position = (4, 1)  # g2's position could change if the game rules say so

    # Update position based on mover's action
    new_position = update_position(position, mover_action)

    # True goal could be influenced by mover_action
    true_goal = 'g1' if mover_action == 'deceive_g2' else 'g2'  # Example logic

    # Update banana counts
    if new_position == g1_position:
        bananas_consumed_at_g1 += action[0]
    if new_position == g2_position:
        bananas_consumed_at_g2 += action[1]

    # Reward calculation
    reward = get_reward(action, true_goal)

    next_state = (new_position, bananas_consumed_at_g1, bananas_consumed_at_g2)
    return next_state, reward


In [14]:
def mover_policy():
    # Randomly decide to deceive about g1 or g2 with a normal action
    if np.random.rand() > 0.5:
        return 'deceive_g1'  # Pretend g1 is the goal but it might be g2
    else:
        return 'deceive_g2'  # Pretend g2 is the goal but it might be g1


In [15]:
import random
def select_action(goal_beliefs, epsilon):
    actions = [(1, 0), (0, 1), (0.5, 0.5), (0.2, 0.8), (0.6, 0.4), (0.8, 0.2), (0.4, 0.6), (0.7, 0.3), (0.3, 0.7)]
    if np.random.rand() < epsilon:  # Explore randomly
        return random.choice(actions)
    else:  # Exploit based on beliefs
        return actions[0] if goal_beliefs[0] > goal_beliefs[1] else actions[1]


In [16]:
def is_terminal(state):
    # Unpacking the state
    position, bananas_consumed_at_g1, bananas_consumed_at_g2 = state

    # Define conditions for game end
    max_bananas_at_g1 = 10  # Maximum bananas that can be consumed at g1
    max_bananas_at_g2 = 10  # Maximum bananas that can be consumed at g2

    # Check if all bananas are consumed at either goal
    if bananas_consumed_at_g1 >= max_bananas_at_g1 or bananas_consumed_at_g2 >= max_bananas_at_g2:
        return True

    # Additional conditions can be added here, such as a maximum number of moves
    # or reaching a particular position on the grid if relevant

    return False  # If none of the terminal conditions are met, continue the game


In [24]:
def monte_carlo(env, select_action, goal_beliefs, epsilon, num_episodes, initial_state):
    V = np.zeros((7, 7))  # Assuming a 7x7 grid for simplicity
    returns = {}

    for _ in range(num_episodes):
        state = initial_state
        episode = []
        while not is_terminal(state):
            mover_action = random.choice(list(movements.keys()))  # Random mover action simulating deception
            action = select_action(goal_beliefs, epsilon)
            next_state, reward = env(state, action, mover_action)  # true_goal is determined inside env
            episode.append((state, action, reward))
            state = next_state

        # Calculate the return and update value estimates
        G = 0
        for state, action, reward in reversed(episode):
            G = reward + G  # Discount factor assumed to be 1 for simplicity
            pos = (state[0][0], state[0][1])
            if pos not in returns:
                returns[pos] = []
            returns[pos].append(G)
            V[pos[0], pos[1]] = np.mean(returns[pos])

    return V


# Example parameters
epsilon = 0.1
goal_beliefs = [0.5, 0.5]
num_episodes = 100
initial_state = ((3, 3), 0, 0)  # Assume starting in the middle of the grid

# Run the Monte Carlo simulation
V = monte_carlo(env, select_action, goal_beliefs, epsilon, num_episodes, initial_state)

print("Estimated Value Function:")
print(V)

Estimated Value Function:
[[316.26953908 335.85185185 338.57648099 342.51303616 367.20746888
  427.14090177 438.58679525]
 [324.15615907 339.02406417 353.68856655 371.17895609 400.20850368
  414.59615385 442.68569311]
 [315.60399637 318.13786929 363.47093023 393.04683841 396.05533905
  389.78422619 430.67795389]
 [292.92301255 275.11565836 325.60015649 378.71903114 386.55907781
  393.90979563 429.31096681]
 [273.1260274  237.75180972 297.82733224 357.26212121 374.14544025
  368.34832317 382.95441176]
 [303.54908775 268.73402256 307.10650888 337.09217661 356.4837037
  370.0755132  380.24793956]
 [296.12982456 284.70373588 291.17899761 316.6726233  332.59900285
  357.46902655 379.48536419]]
